# Tutorial: Mastering Google Cloud Speech APIs with `speech.py`

Welcome to this comprehensive tutorial on using the `speech.py` module! This module provides a convenient Python wrapper for interacting with Google Cloud's powerful Speech-to-Text (STT) v1 and v2, and Text-to-Speech (TTS) APIs.

With `speech.py`, you can easily:
- Transcribe audio files into text.
- Leverage advanced STT features like speaker diarization and language translation (via v2).
- Synthesize natural-sounding speech from text input.

This notebook will guide you through the setup, core functionalities, and practical examples for each component of the `speech.py` module.

## 🚀 Setup

First, we need to install the necessary Google Cloud client libraries. The `speech.py` module also relies on a `conidk.core.base` module for authentication and configuration. Since `conidk` is an internal dependency, we'll create a minimal mock implementation of `base.Auth` and `base.Config` to enable this tutorial to run standalone. This mock will utilize Google Application Default Credentials (ADC), making it compatible with Colab's authentication or local `gcloud auth` setups.

In [ ]:
# Install necessary libraries
!pip install google-cloud-speech google-cloud-texttospeech --upgrade

import os
import enum
import uuid
from typing import Optional

from google.cloud import texttospeech
from google.cloud import speech_v1
from google.cloud.speech_v1 import types as types_v1
from google.cloud import speech_v2
from google.cloud.speech_v2 import types as types_v2

# Imports for mock `conidk.core.base`
from google.oauth2 import service_account
from google.auth import credentials as auth_credentials
from google.api_core.client_options import ClientOptions
import google.auth

# Mock conidk.core.base for demonstration purposes
# In a real environment, you would have 'conidk' installed and configured.
class MockAuth:
    """Mock Auth class to provide Google Cloud credentials."""
    def __init__(self, credentials_path: Optional[str] = None):
        self.creds = None # Initialize to None
        if credentials_path:
            # If a service account key path is provided
            self.creds = service_account.Credentials.from_service_account_file(credentials_path)
            print(f"Using service account credentials from: {credentials_path}")
        else:
            # Attempt to use default credentials (e.g., Colab, gcloud auth, env var GOOGLE_APPLICATION_CREDENTIALS)
            try:
                # For Google Colab, authenticate the user
                from google.colab import auth as colab_auth
                colab_auth.authenticate_user()
                # Colab auth sets up ADC globally, often no need to pass explicit creds to client
                # However, some clients might still benefit from explicit credentials, so we fetch default if possible.
                self.creds, _ = google.auth.default()
                print("Authenticated with Colab. Using default application credentials.")
            except ImportError:
                # Fallback for local execution or other environments
                # Assumes GOOGLE_APPLICATION_CREDENTIALS is set or `gcloud auth application-default login` is done
                self.creds, project = google.auth.default()
                print(f"Using default application credentials. Project: {project}")

class MockConfig:
    """Mock Config class to provide client options (e.g., API endpoints)."""
    def set_speech_endpoint(self):
        # Default endpoint for Speech-to-Text. Can be customized if needed.
        return ClientOptions(api_endpoint="speech.googleapis.com")

    def set_texttospeech_endpoint(self):
        # Default endpoint for Text-to-Speech. Can be customized if needed.
        return ClientOptions(api_endpoint="texttospeech.googleapis.com")

# Create a mock 'base' module structure so `speech.py` can import it.
class MockBaseModule:
    Auth = MockAuth
    Config = MockConfig

base = MockBaseModule()

print("Libraries installed and mock `conidk.core.base` setup complete.")


### ☁️ Google Cloud Configuration
You'll need a Google Cloud Project with the **Cloud Speech-to-Text API** and **Cloud Text-to-Speech API** enabled. Replace `YOUR_PROJECT_ID` with your actual Google Cloud Project ID.

For Speech-to-Text V2, batch recognition results are written to a Google Cloud Storage (GCS) bucket. Make sure to replace `YOUR_GCS_BUCKET_FOR_V2_OUTPUT` with a valid GCS bucket URI where your service account has write permissions.

In [ ]:
# --- Google Cloud Project Configuration --- 
PROJECT_ID = "YOUR_PROJECT_ID" # <<< REPLACE THIS WITH YOUR GOOGLE CLOUD PROJECT ID
GCS_AUDIO_FILE_URI = "gs://cloud-samples-data/speech/brooklyn_bridge.flac" # Example GCS URI for an audio file
GCS_V2_OUTPUT_BUCKET = "gs://YOUR_GCS_BUCKET_FOR_V2_OUTPUT/" # <<< REPLACE THIS with a GCS bucket for V2 outputs

# Validation checks
if PROJECT_ID == "YOUR_PROJECT_ID":
    raise ValueError("Please replace 'YOUR_PROJECT_ID' with your actual Google Cloud Project ID.")
if GCS_V2_OUTPUT_BUCKET == "gs://YOUR_GCS_BUCKET_FOR_V2_OUTPUT/":
    print("Warning: For Speech-to-Text V2 batch recognition, you'll need a GCS bucket for output. Please replace 'YOUR_GCS_BUCKET_FOR_V2_OUTPUT' for a real scenario.")

print(f"Google Cloud Project ID: {PROJECT_ID}")


## 📝 The `speech.py` Module Source Code
Below is the complete source code for the `speech.py` module. Running this cell will define the `AudioChannels`, `Encodings`, `V1`, `V2`, and `TextToSpeech` classes, making them available for use in the subsequent examples.

In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""This module provides a wrapper for Google Cloud Speech-to-Text and Text-to-Speech APIs."""

# The required imports from google.cloud and standard libraries are already present at the top of the notebook.
# The 'base' module is now our mocked 'base' object.


class AudioChannels(enum.IntEnum):
    """Enum for audio channel options."""
    MONO = 1
    STEREO = 2

class Encodings(enum.Enum):
    """Enum for audio encoding types, simplifying options for the user."""
    LINEAR16 = types_v1.RecognitionConfig.AudioEncoding.LINEAR16
    FLAC = types_v1.RecognitionConfig.AudioEncoding.FLAC
    MULAW = types_v1.RecognitionConfig.AudioEncoding.MULAW
    AMR = types_v1.RecognitionConfig.AudioEncoding.AMR
    AMR_WB = types_v1.RecognitionConfig.AudioEncoding.AMR_WB
    OGG_OPUS = types_v1.RecognitionConfig.AudioEncoding.OGG_OPUS
    WEBM_OPUS = types_v1.RecognitionConfig.AudioEncoding.WEBM_OPUS

class V1:
    """A wrapper for the Google Cloud Speech-to-Text v1 API.

    Attributes:
        auth: An authentication object.
        config: A configuration object.
        client: A Speech-to-Text v1 client.
    """

    def __init__(
        self,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ):
        """Initializes the V1 wrapper.

        Args:
            auth: An authentication object.
            config: A configuration object.
        """
        self.auth = auth or base.Auth()
        self.config = config or base.Config()

        self.client = speech_v1.SpeechClient(
            credentials=self.auth.creds,
            client_options=self.config.set_speech_endpoint()
        )

    def create_transcription(
        self,
        audio_file_path: str,
        language: str = "en-US",
        audio_type: AudioChannels = AudioChannels.STEREO,
        model: str = "latest_short",
        sample_rate: int = 8000,
        encoding: Encodings = Encodings.MULAW,
    ) -> types_v1.LongRunningRecognizeResponse:

        """Transcribes a long audio file using the Speech-to-Text v1 API.

        Args:
            audio_file_path: The Google Cloud Storage URI of the audio file.
            language: The language of the speech in the audio.
            audio_type: The number of channels in the audio.
            model: The recognition model to use.
            sample_rate: The sample rate of the audio.
            encoding: The encoding of the audio.

        Returns:
            A LongRunningRecognizeResponse object containing the transcription results.
        """
        operation = self.client.long_running_recognize(
            config=self._setup_recognition_config(
                channels=audio_type.value,
                audio_type=audio_type,
                language=language,
                encoding=encoding,
                model=model,
                sample_rate_hertz=sample_rate,
            ),
            audio=speech_v1.RecognitionAudio(uri=audio_file_path),
        )
        return operation.result()

    def _setup_recognition_config(
        self,
        language: str,
        encoding: Encodings,
        model: str,
        sample_rate_hertz: int,
        audio_type: AudioChannels = AudioChannels.STEREO,
        channels: Optional[int] = None,
    ) -> types_v1.RecognitionConfig:
        """Sets up the recognition configuration for a transcription request.

        Args:
            language: The language of the speech in the audio.
            encoding: The encoding of the audio.
            model: The recognition model to use.
            sample_rate_hertz: The sample rate of the audio.
            audio_type: The number of channels in the audio.
            channels: The number of audio channels.

        Returns:
            A RecognitionConfig object.
        """

        if channels:
            channel_count = channels
        else:
            channel_count = audio_type.value

        config = speech_v1.RecognitionConfig(
            audio_channel_count=channel_count,
            sample_rate_hertz=sample_rate_hertz,
            language_code=language,
            encoding=encoding.value,
            model=model,
            enable_automatic_punctuation=True,
            enable_word_time_offsets=True,
        )

        if channels == 1:
            config.diarization_config = speech_v1.SpeakerDiarizationConfig(
                enable_speaker_diarization=True,
                min_speaker_count=1,
                max_speaker_count=2,
            )

        return config

class V2:
    """A wrapper for the Google Cloud Speech-to-Text v2 API.

    Attributes:
        project_id: The Google Cloud project ID.
        diarization: Whether to enable speaker diarization.
        auto_decoding: Whether to enable auto-decoding.
        model: The recognition model to use.
        location: The location of the Speech-to-Text service.
        auth: An authentication object.
        translation: Whether to enable translation.
        config: A configuration object.
        tmp_storage: A temporary storage location for transcription results.
        language_code: The language of the speech in the audio.
        translate_languange: The target language for translation.
        client: A Speech-to-Text v2 client.
    """

    def __init__(
        self,
        project_id: str,
        diarization: bool = False,
        auto_decoding: bool = True,
        model: Optional[str] = "long",
        location: Optional[str] = None,
        auth: Optional[base.Auth] = None,
        translation: Optional[bool] = False,
        config: Optional[base.Config] = None,
        tmp_storage: Optional[str] = None,
        language_code: Optional[str] = "en-US",
        translate_languange: Optional[str] = None,
    ):
        """Initializes the V2 wrapper.

        Args:
            project_id: The Google Cloud project ID.
            diarization: Whether to enable speaker diarization.
            auto_decoding: Whether to enable auto-decoding.
            model: The recognition model to use.
            location: The location of the Speech-to-Text service.
            auth: An authentication object.
            translation: Whether to enable translation.
            config: A configuration object.
            tmp_storage: A temporary storage location for transcription results.
            language_code: The language of the speech in the audio.
            translate_languange: The target language for translation.
        """

        self.auth = auth or base.Auth()
        self.config = config or base.Config()
        self.project_id = project_id
        self.diarization = diarization
        self.auto_decoding = auto_decoding
        self.model = model
        self.location = location if location else "global" # Use 'global' or a specific region like 'us-central1'
        self.translation = translation
        self.tmp_storage = tmp_storage
        self.language_code = language_code
        self.translate_languange = translate_languange

        self.client = speech_v2.SpeechClient(
            credentials=self.auth.creds,
            client_options=self.config.set_speech_endpoint()
        )

    def _generate_id(self, default_name: str) -> str:
        """Generates a unique ID for a resource.
        
        Args:
            default_name: The default name to use as a prefix.
            
        Returns:
            A unique ID string.
        """
        return default_name + "-" + str(uuid.uuid4())[:15].lower().replace("-", "")

    def _setup_recognizer(
        self,
        name: str,
    ) -> types_v2.Recognizer:
        """Sets up a recognizer for a transcription request.
        
        Args:
            name: The display name for the recognizer.
            
        Returns:
            A Recognizer object.
        """
        recognizer = types_v2.Recognizer(
            display_name=name,
            default_recognition_config=types_v2.RecognitionConfig(
                model=self.model,
                language_codes=[self.language_code],
                features=types_v2.RecognitionFeatures(
                    profanity_filter=True,
                    enable_word_time_offsets=True,
                    enable_word_confidence=True,
                    enable_automatic_punctuation=True,
                    enable_spoken_punctuation=True,
                    enable_spoken_emojis=True,
                ),
            ),
        )

        if self.diarization:
            # Diarization config needs to be added to features
            recognizer.default_recognition_config.features.diarization_config = (
                types_v2.SpeakerDiarizationConfig(
                    min_speaker_count = 1,
                    max_speaker_count = 2
                )
            )

        if self.auto_decoding:
            recognizer.default_recognition_config.auto_decoding_config = (
                types_v2.AutoDetectDecodingConfig()
            )
        if self.translation:
            if self.translate_languange:
                recognizer.default_recognition_config.translation_config = (
                    types_v2.TranslationConfig(target_language=self.translate_languange)
                )
            else:
                raise AttributeError("Translate language is not set when translation is enabled.")
        return recognizer

    def create_recognizer(
        self,
        name: str = "default-insights-recognizer"
    ) -> str:
        """Creates a Recognizer resource in the Speech-to-Text v2 API.

        Args:
            name: The display name for the recognizer.

        Returns:
            The resource name of the created recognizer.
        """

        # Ensure location is set to avoid errors if using specific regions for V2
        if not self.location:
            raise ValueError("Location must be set for V2 Speech-to-Text API.")
            
        operation = self.client.create_recognizer(
            parent=f"projects/{self.project_id}/locations/{self.location}",
            recognizer_id=self._generate_id(name),
            recognizer=self._setup_recognizer(name),
        )
        result = operation.result()
        return result.name

    def create_transcription(
        self,
        audio_file_path: str,
        recognizer_path: Optional[str] = None,
    ) -> types_v2.BatchRecognizeResponse:
        """Transcribes a long audio file using the Speech-to-Text v2 API.

        Args:
            audio_file_path: The Google Cloud Storage URI of the audio file.
            recognizer_path: The resource name of the recognizer to use.

        Returns:
            A long-running operation object for the batch recognition request.
        """

        if not recognizer_path:
            print("No recognizer_path provided, creating a new one...")
            recognizer_path = self.create_recognizer()
            print(f"Created recognizer: {recognizer_path}")

        if not self.tmp_storage:
            raise ValueError("tmp_storage (GCS bucket for output) must be set for V2 batch recognition.")

        print(f"Starting batch recognition for {audio_file_path} using recognizer {recognizer_path}...")
        operation = self.client.batch_recognize(
            request = types_v2.BatchRecognizeRequest(
                recognizer=recognizer_path,
                files=[types_v2.BatchRecognizeFileMetadata(uri=audio_file_path)],
                recognition_output_config=types_v2.RecognitionOutputConfig(
                    gcs_output_config=types_v2.GcsOutputConfig(uri=self.tmp_storage),
                )

            )
        )

        print("Batch recognition operation submitted. Waiting for results (this may take a while)...")
        return operation.result() # This will block until the operation is complete

class TextToSpeech:
    """A wrapper for the Google Cloud Text-to-Speech API.

    Attributes:
        project_id: The Google Cloud project ID.
        auth: An authentication object.
        config: A configuration object.
        client: A Text-to-Speech client.
    """

    def __init__(
        self,
        project_id: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    )->None:
        """Initializes the TextToSpeech wrapper.

        Args:
            project_id: The Google Cloud project ID.
            auth: An authentication object.
            config: A configuration object.
        """
        self.auth = auth or base.Auth()
        self.config = config or base.Config()
        self.project_id = project_id

        self.client = texttospeech.TextToSpeechClient(
            credentials=self.auth.creds,
            client_options=self.config.set_texttospeech_endpoint()
        )

    def synthesize(
        self,
        text: str,
        voice: str = "en-US-Wavenet-D",
        language_code: Optional[str]= "en-US",
        sample_rate_hertz: int = 32000
    ):
        """Synthesizes speech from a string of text.

        Args:
            text: The input text to synthesize.
            voice: The name of the voice to use for synthesis.
            language_code: The language of the voice.
            sample_rate_hertz: The desired sample rate of the audio.

        Returns:
            The synthesized audio content as bytes.
        """

        synthesis_input = texttospeech.SynthesisInput(text=text)

        voice = texttospeech.VoiceSelectionParams(
            language_code=language_code,
            name=voice
        )
        #TO-DO: Support multiple encoding selection
        response = self.client.synthesize_speech(
            input=synthesis_input,
            voice=voice,
            audio_config=texttospeech.AudioConfig(
                audio_encoding=texttospeech.AudioEncoding.LINEAR16,
                sample_rate_hertz=sample_rate_hertz
            )
        )
        return response.audio_content


## 🎙️ Speech-to-Text V1: Basic Transcription
The `V1` class provides a wrapper for the Google Cloud Speech-to-Text v1 API. This API is well-suited for transcribing long audio files stored in Google Cloud Storage and offers a straightforward configuration model.

### Key Parameters for V1 Transcription:
- `audio_file_path`: A Google Cloud Storage URI (e.g., `gs://bucket/path/to/file.flac`).
- `language`: The language code of the speech (e.g., `"en-US"`).
- `audio_type`: The number of channels in the audio, using `AudioChannels.MONO` or `AudioChannels.STEREO`.
- `model`: The recognition model to use (`"latest_short"`, `"video"`, `"phone_call"`, etc.).
- `sample_rate`: The audio's sample rate in Hertz.
- `encoding`: The audio encoding type, using the `Encodings` enum (e.g., `Encodings.FLAC`).

In [ ]:
print("\n--- Demonstrating Speech-to-Text V1 ---\n")

# Initialize the V1 client
stt_v1_client = V1(auth=base.Auth(), config=base.Config())

# Perform transcription
print(f"Transcribing audio from: {GCS_AUDIO_FILE_URI}")
try:
    v1_response = stt_v1_client.create_transcription(
        audio_file_path=GCS_AUDIO_FILE_URI,
        language="en-US",
        audio_type=AudioChannels.MONO, # Assuming the example audio is mono
        model="video", # A versatile model for general audio
        sample_rate=16000, # Common sample rate, adjust if your audio differs
        encoding=Encodings.FLAC # Matching the example FLAC file
    )

    print("\nV1 Transcription Results:")
    for i, result in enumerate(v1_response.results):
        print(f"Segment {i+1}:")
        print(f"  Transcript: {result.alternatives[0].transcript}")
        print(f"  Confidence: {result.alternatives[0].confidence:.2f}")
        if result.alternatives[0].words:
            # Display first and last word with time offsets
            first_word = result.alternatives[0].words[0]
            last_word = result.alternatives[0].words[-1]
            print(f"  First word: '{first_word.word}' (Start: {first_word.start_time.seconds}s.{first_word.start_time.nanos // 1000000:03d}ms)")
            print(f"  Last word: '{last_word.word}' (End: {last_word.end_time.seconds}s.{last_word.end_time.nanos // 1000000:03d}ms)")
        
        # Diarization results are often detailed and might need further parsing.
        # This check confirms if diarization was enabled and results are present.
        if result.speaker_diarization_prediction:
            print("  Speaker diarization data available (check full result object for details).")

except Exception as e:
    print(f"\nError during V1 transcription: {e}")
    print("Please ensure:")
    print("  1. The GCS_AUDIO_FILE_URI is correct and the audio file exists and is accessible.")
    print("  2. Your credentials have permission to access the GCS bucket.")
    print("  3. The Google Cloud Speech-to-Text API (v1) is enabled for your project.")
    print("  4. Audio parameters (encoding, sample rate, channels) match the actual audio file.")


## 🚀 Speech-to-Text V2: Advanced Recognition with Recognizers
The `V2` class utilizes the newer Google Cloud Speech-to-Text v2 API, which offers more advanced features and improved resource management through the concept of **Recognizers**. Recognizers are reusable configurations that encapsulate all settings for a transcription job, promoting consistency and reducing redundant setup.

### Key Features of V2:
- **Recognizers**: Define and reuse recognition configurations.
- **Batch Recognition**: Efficiently transcribe multiple audio files by specifying a list of GCS URIs.
- **Diarization**: Accurately identify and separate different speakers in an audio file.
- **Translation**: Transcribe audio and simultaneously translate the results into another language.
- **Auto-Decoding**: Automatically detect audio encoding and sample rate, simplifying configuration.
- **Output to GCS**: Detailed transcription results are written to a specified GCS bucket as JSON files, enabling easy post-processing.

In [ ]:
print("\n--- Demonstrating Speech-to-Text V2 ---\n")

# Ensure the GCS bucket for V2 output is set
if GCS_V2_OUTPUT_BUCKET == "gs://YOUR_GCS_BUCKET_FOR_V2_OUTPUT/":
    raise ValueError("Please replace 'YOUR_GCS_BUCKET_FOR_V2_OUTPUT' with a valid GCS bucket for V2 batch recognition outputs.")

# Initialize the V2 client with project ID, location, and GCS output bucket
stt_v2_client = V2(
    project_id=PROJECT_ID,
    location="global", # Or a specific region like "us-central1" for regional recognizers
    diarization=True, # Enable speaker diarization
    auto_decoding=True, # Automatically detect encoding/sample rate
    model="telephony", # Using a model suitable for phone calls, for example
    tmp_storage=GCS_V2_OUTPUT_BUCKET, # GCS bucket for batch recognition output results
    language_code="en-US",
    # translation=True, # Uncomment to enable translation
    # translate_languange="es", # Set target language if translation is True
    auth=base.Auth(), 
    config=base.Config()
)

try:
    # 1. Create a Recognizer
    # The recognizer encapsulates configuration for transcription and is a reusable resource.
    print("Creating a Recognizer resource...")
    recognizer_name = stt_v2_client.create_recognizer(name="my-colab-recognizer")
    print(f"Recognizer created: {recognizer_name}")

    # 2. Perform Batch Transcription using the Recognizer
    print(f"Starting V2 batch transcription for {GCS_AUDIO_FILE_URI} using recognizer {recognizer_name}...")
    v2_batch_response = stt_v2_client.create_transcription(
        audio_file_path=GCS_AUDIO_FILE_URI,
        recognizer_path=recognizer_name # Use the created recognizer
    )

    print("\nV2 Batch Transcription Operation Submitted:")
    # V2 batch results are asynchronously written to GCS. The response provides URIs to the output.
    for file_result in v2_batch_response.results:
        print(f"  Processed Audio file: {file_result.uri}")
        print(f"  Transcription output GCS URI: {file_result.transcript_output_config.gcs_output_uri}")
    
    print("\nNote: For V2 batch recognition, the detailed transcription results (including diarization, word timings, etc.) are stored as JSON files in the specified GCS output bucket (`tmp_storage`). You would need to download and parse these files to view the full results. Example:")
    print("  `from google.cloud import storage; storage_client = storage.Client(project=PROJECT_ID);`")
    print("  `bucket_name = GCS_V2_OUTPUT_BUCKET.split('gs://')[1].split('/')[0]`")
    print("  `blob_name = '/'.join(file_result.transcript_output_config.gcs_output_uri.split('gs://')[1].split('/')[1:])`")
    print("  `blob = storage_client.bucket(bucket_name).blob(blob_name); content = blob.download_as_text(); import json; print(json.loads(content))`")

except Exception as e:
    print(f"\nError during V2 transcription: {e}")
    print("Please ensure:")
    print("  1. Your PROJECT_ID and GCS_V2_OUTPUT_BUCKET are correct and exist.")
    print("  2. The Google Cloud Speech-to-Text API (v2) is enabled for your project.")
    print("  3. Your credentials have appropriate GCS write permissions to GCS_V2_OUTPUT_BUCKET and Speech API permissions.")
    print("  4. The specified GCS_AUDIO_FILE_URI is correct and accessible.")


## 🗣️ Text-to-Speech: Synthesize Audio from Text
The `TextToSpeech` class allows you to convert written text into natural-sounding speech using Google Cloud's Text-to-Speech API. The API supports various voices, languages, and audio configurations. The `synthesize` method returns the audio content as raw bytes, which can then be saved to a file or played directly.

### Key Parameters for TTS Synthesis:
- `text`: The input string that you want to synthesize into speech.
- `voice`: The name of the voice to use (e.g., `"en-US-Wavenet-D"`, `"en-GB-Standard-A"`). You can find a comprehensive list of available voices and their attributes in the [Google Cloud Text-to-Speech documentation](https://cloud.google.com/text-to-speech/docs/voices).
- `language_code`: The BCP-47 language code for the voice (e.g., `"en-US"`, `"es-ES"`).
- `sample_rate_hertz`: The desired audio sample rate in Hertz (e.g., `24000`, `32000`, `48000`).

In [ ]:
print("\n--- Demonstrating Text-to-Speech ---\n")

# Initialize the TextToSpeech client
tts_client = TextToSpeech(
    project_id=PROJECT_ID,
    auth=base.Auth(), 
    config=base.Config()
)

text_to_synthesize = "Hello from the Google Cloud Text-to-Speech API! This is a demonstration of converting text into spoken audio using a Wavenet voice."
output_filename = "output_speech.wav"

print(f"Synthesizing text: '{text_to_synthesize}'")
try:
    audio_content = tts_client.synthesize(
        text=text_to_synthesize,
        voice="en-US-Wavenet-F", # Example Wavenet voice
        language_code="en-US",
        sample_rate_hertz=24000
    )

    # Save the audio content to a WAV file
    with open(output_filename, "wb") as out:
        out.write(audio_content)
    print(f"Audio content successfully written to {output_filename}")

    # Optional: Play the audio directly in Colab
    try:
        from IPython.display import Audio, display
        print("Playing synthesized audio...")
        display(Audio(output_filename, autoplay=True))
    except ImportError:
        print("IPython.display.Audio not available. You can download and play the WAV file locally.")

except Exception as e:
    print(f"\nError during Text-to-Speech synthesis: {e}")
    print("Please ensure the Google Cloud Text-to-Speech API is enabled for your project.")


## 🎉 Conclusion
You've successfully navigated the features of the `speech.py` module, demonstrating how to interact with Google Cloud's Speech-to-Text v1, Speech-to-Text v2, and Text-to-Speech APIs. By following this tutorial, you've learned to:

- Set up your environment, including mocking internal dependencies and configuring Google Cloud project details.
- Perform basic audio transcription using the `V1` API.
- Utilize the advanced features of the `V2` API, including creating and using Recognizers for batch transcription, and understanding its output mechanism.
- Synthesize text into high-quality speech using the `TextToSpeech` API and play the generated audio.

Remember to replace all placeholder values like `YOUR_PROJECT_ID` and `YOUR_GCS_BUCKET_FOR_V2_OUTPUT` with your specific Google Cloud resources. The Google Cloud documentation for [Speech-to-Text](https://cloud.google.com/speech-to-text/docs) and [Text-to-Speech](https://cloud.google.com/text-to-speech/docs) are excellent resources for exploring more advanced options, voice types, models, and best practices.